In [27]:
!pip install pypdf faiss-cpu sentence-transformers langchain gpt4all langchain-community rank_bm25

In [28]:
!pip install -q transformers torch rouge-score nltk pandas

In [30]:
# ===== 1. Import Libraries =====
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import GPT4AllEmbeddings
from langchain.vectorstores import FAISS
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import re
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from nltk import pos_tag, word_tokenize

# ===== 2. Download NLTK resources =====
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [31]:
# ===== 3. Loading Data =====
pdf_paths = "/kaggle/input/sherlock-homes/Sir Arthur Conan Doyle - Complete Sherlock Holmes (1960)_removed.pdf"
loader = PyPDFLoader(pdf_paths)
documents = loader.load()

In [32]:
# ===== 4. Define Chunking Strategy =====
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
texts = text_splitter.split_documents(documents)

In [ ]:
# ===== 5. Define Retrieval Strategy =====
embeddings = GPT4AllEmbeddings()
db = FAISS.from_documents(texts, embeddings)
bm25 = BM25Retriever.from_documents(texts)
bm25.k = 3
ensemble = EnsembleRetriever(retrievers=[db.as_retriever(search_kwargs={"k": 3}), bm25], weights=[0.5, 0.5])

# ===== 6. Retriever Output =====
question = "Who is Sherlock Holmes' assistant?"
retrieved_docs = ensemble.get_relevant_documents(question)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])



In [ ]:
# ===== 7. Text Preprocessing Setup =====
STOPWORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def nltk_pos_to_wordnet(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def preprocess_text(text):
    # Tokenization
    tokens = word_tokenize(text)
    
    # Lowercasing
    tokens = [t.lower() for t in tokens]
    
    # Strip punctuation
    tokens = [re.sub(r'[^\w\s]', '', t) for t in tokens]
    tokens = [t for t in tokens if t]
    
    # Remove stopwords
    tokens = [t for t in tokens if t.lower() not in STOPWORDS]
    
    # Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    
    # Lemmatization
    pos_tags = pos_tag(tokens)
    tokens = [lemmatizer.lemmatize(w, nltk_pos_to_wordnet(p)) for w, p in pos_tags]
    
    return " ".join(tokens)

# Preprocess question and context
processed_question = preprocess_text(question)
processed_context = preprocess_text(context)


In [ ]:
# ===== 8. Load Model and define its parameters =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "google/flan-t5-xxl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

generation_args = {
    "max_length": 150,
    "num_beams": 4,
    "top_k": 40,
    "top_p": 0.95
}

# ===== 9. Generate and Print Output =====
input_text = f"question: {processed_question} context: {processed_context}"
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
outputs = model.generate(inputs["input_ids"], **generation_args)
model_output = tokenizer.decode(outputs[0], skip_special_tokens=True)



In [ ]:
print("\n" + "="*80)
print("Model Output")
print("="*80)
print(model_output)